In [1]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import torch
from ollama import Client

In [2]:
def load_text_from_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        text = file.read()
    return text

In [3]:
def load_and_chunk_text(text):
    # Initialize the text splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1024,
        chunk_overlap=200,
        length_function=len,  # Use character count
    )

    # Split the text into chunks
    chunks = text_splitter.split_text(text)

    # Print the chunks
    for i, chunk in enumerate(chunks):
        print(f"Chunk {i+1}:\n{chunk}\n")

    return chunks

In [4]:
def embed_chunks(chunks):
    # Load the embedding model
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2', device="cuda")

    # Embed the chunks
    chunk_embeddings = embedding_model.encode(chunks, device="cuda")

    # Print the shape of the embeddings
    print(f"Embeddings shape: {chunk_embeddings.shape}")

    return embedding_model, chunk_embeddings

# Step 4: Store embeddings in FAISS
def create_faiss_index(chunk_embeddings):
    # Create a FAISS index on GPU
    dimension = chunk_embeddings.shape[1]  # Embedding dimension
    res = faiss.StandardGpuResources()  # Use GPU resources
    index = faiss.IndexFlatL2(dimension)  # L2 distance for similarity search
    gpu_index = faiss.index_cpu_to_gpu(res, 0, index)  # Move index to GPU

    # Add embeddings to the index
    gpu_index.add(np.array(chunk_embeddings))

    print(f"FAISS index contains {gpu_index.ntotal} embeddings.")

    return gpu_index

In [5]:
def answer_question(question, embedding_model, index, chunks, ollama_client, top_k=3):
    # Embed the question
    question_embedding = embedding_model.encode([question], device="cuda")

    # Retrieve the most relevant chunks
    distances, indices = index.search(np.array(question_embedding), top_k)
    relevant_chunks = [chunks[i] for i in indices[0]]

    # Format the prompt
    prompt = f"""Based on the following context, please provide a concise answer to the question.

Context:
{' '.join(relevant_chunks)}

Question: {question}

Answer: Let me answer based on the provided context."""

    # Generate answer using local Ollama model
    response = ollama_client.generate(
        model='deepseek-r1',  # Make sure you have this model pulled locally
        prompt=prompt,
    )
    
    return response['response']

In [14]:
def input_flow():
    try:
        text = load_text_from_file("resmigazete2sf.txt")

        print("Step 1: Loading and chunking text...")
        chunks = load_and_chunk_text(text)

        print("Step 2: Embedding chunks...")
        embedding_model, chunk_embeddings = embed_chunks(chunks)

        print("Step 3: Creating FAISS index...")
        index = create_faiss_index(chunk_embeddings)

        return embedding_model, index, chunks  # Return required variables

    except Exception as e:
        print(f"An error occurred: {str(e)}")
        return None, None, None  # Handle failure cases

In [15]:
def output_flow(embedding_model, index, chunks):  # Accept variables as parameters
    try:
        if embedding_model is None or index is None or chunks is None:
            print("Error: Required data is missing.")
            return
        
        ollama_client = Client()

        # Step 4: Answer a question
        question = "What is the main topic of the text?"
        print(f"\nQuestion: {question}")
        answer = answer_question(question, embedding_model, index, chunks, ollama_client)
        print(f"\nAnswer: {answer}")

    except Exception as e:
        print(f"An error occurred: {str(e)}")

In [16]:
embedding_model, index, chunks = input_flow()

Step 1: Loading and chunking text...
Chunk 1:
T.C.
Resmî Gazete
Cumhurbaşkanlığı Genel Sekreterliği Hukuk ve Mevzuat Genel Müdürlüğünce Yayımlanır
14 Ocak 2025 SALI
Sayı : 32782
YÜRÜTME VE İDARE BÖLÜMÜ
YÖNETMELİKLER
Çevre, Şehircilik ve İklim Değişikliği Bakanlığından:
ENDÜSTRIYEL EMİSYONLARIN YÖNETİMİ YÖNETMELİĞİ BİRİNCİ BÖLÜM Başlangıç Hükümleri
Amaç
MADDE 1- (1) Bu Yönetmeliğin amacı; çevrenin ve insan sağlığının bütüncül olarak korunması için sıfır kirlilik hedefleri doğrultusunda entegre kirlilik önleme ve kontrol yakla- şımıyla hava, su, toprak, gürültü ve koku kirliliğine neden olan sanayi kaynaklı emisyonları ve atık oluşumunu kaynağında önlemek ve azaltmak ile kaynakları verimli kullanmak için sa- nayide yeşil dönüşüme, döngüsel ekonomiye ve karbonsuzlaşmaya yönelik idari ve teknik usul ve esasları düzenlemektir.
Kapsam
MADDE 2- (1) Bu Yönetmelik, EK-1 ve EK-2'de yer alan faaliyetlerin gerçekleştiril- diği işletmeleri kapsar.

Chunk 2:
Kapsam
MADDE 2- (1) Bu Yönetmelik, EK-1 v

In [17]:
output_flow(embedding_model, index, chunks)  # Pass necessary arguments


Question: What is the main topic of the text?

Answer: <think>
Okay, I need to figure out the main topic of the text provided. The user has given a bunch of definitions related to environmental and occupational health regulations in Turkey. Let's look at each point.

First, "İşletme" refers to business or operations. Then there are terms like "işletmeci," which is an authorized person operating a facility under certain permits. "Kirlilik" translates to pollution control, which makes sense as it deals with environmental regulations. 

"Maddeler" means substances here, and "emisyon" is emissions. The context talks about controlling these emissions into air, water, or soil. So the focus is on managing harmful substances released from industrial activities.

The terms like ESD (Emisyon Sınır Değerleri) indicate specific concentration levels that need to be maintained to prevent environmental harm. "Değişiklik raporu" and "e-SYD Sistemi" suggest process changes and digital systems in handl